# BillVault FINAL APK

**Run: Cell 1 → 2 → 3 → 4 → 5**

- Gemini AI (auto model discovery)
- Electricity, Property, Water, LIC bills
- Google Sheets + Drive upload working


In [ ]:
# BillVault FINAL APK — Step 1: Gemini API Key
# aistudio.google.com → Get API Key → Create → Copy → paste below
import urllib.request, json, os

MY_KEY = "YOUR_API_KEY"  # <── paste here

url = "https://generativelanguage.googleapis.com/v1beta/models?key=" + MY_KEY
try:
    with urllib.request.urlopen(urllib.request.Request(url), timeout=20) as r:
        data = json.loads(r.read())
    models = [m["name"].replace("models/","") for m in data.get("models",[])
              if "generateContent" in m.get("supportedGenerationMethods",[])
              and ("flash" in m["name"] or "pro" in m["name"])
              and not any(x in m["name"] for x in ["tts","embed","image"])]
    models.sort(key=lambda n: 3 if "2.5" in n else 2 if "2.0" in n else 1, reverse=True)
    print("Key valid! Primary model:", models[0] if models else "none")
    os.environ["GEMINI_KEY"] = MY_KEY
    print("Ab Cell 2 run karo")
except Exception as e:
    print("Error:", e)


In [ ]:
# Step 2: HTML file generate karo
import os, base64

HTML_B64 = "..."

os.makedirs("/content/BillVaultApp/app/src/main/assets", exist_ok=True)
out = "/content/BillVaultApp/app/src/main/assets/index.html"
with open(out, "wb") as f:
    f.write(base64.b64decode(HTML_B64))
print(f"index.html written: {os.path.getsize(out)/1024:.1f} KB")
print("Ab Cell 3 run karo")


In [ ]:
%%bash
# Step 3: Android project setup (sirf pehli baar run karo)
set -e
SDK=/content/android-sdk
APP=/content/BillVaultApp

if [ ! -d "$SDK/cmdline-tools/latest/bin" ]; then
  echo "Downloading Android SDK..."
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/ct.zip
  mkdir -p $SDK/cmdline-tools
  unzip -q /tmp/ct.zip -d /tmp/cx
  mv /tmp/cx/cmdline-tools $SDK/cmdline-tools/latest
  echo "SDK ready"
fi

export ANDROID_HOME=$SDK ANDROID_SDK_ROOT=$SDK
export PATH=$PATH:$SDK/cmdline-tools/latest/bin:$SDK/build-tools/33.0.0

if [ ! -d "$SDK/platforms/android-33" ]; then
  echo "Installing android-33..."
  yes | sdkmanager --sdk_root=$SDK "platforms;android-33" "build-tools;33.0.0" > /dev/null 2>&1
  yes | sdkmanager --sdk_root=$SDK --licenses > /dev/null 2>&1
  echo "Platform installed"
fi

mkdir -p $APP/app/src/main/java/com/billvault
mkdir -p $APP/app/src/main/res/values
mkdir -p $APP/gradle/wrapper

cat > $APP/app/src/main/AndroidManifest.xml << 'XML'
<?xml version="1.0" encoding="utf-8"?>
<manifest xmlns:android="http://schemas.android.com/apk/res/android" package="com.billvault">
  <uses-permission android:name="android.permission.INTERNET"/>
  <uses-permission android:name="android.permission.READ_EXTERNAL_STORAGE"/>
  <application android:allowBackup="true" android:label="BillVault"
      android:theme="@android:style/Theme.NoTitleBar.Fullscreen">
    <activity android:name=".MainActivity" android:exported="true"
        android:configChanges="orientation|screenSize|keyboardHidden">
      <intent-filter>
        <action android:name="android.intent.action.MAIN"/>
        <category android:name="android.intent.category.LAUNCHER"/>
      </intent-filter>
    </activity>
  </application>
</manifest>
XML

cat > $APP/app/src/main/java/com/billvault/MainActivity.java << 'JAVA'
package com.billvault;
import android.app.Activity;
import android.content.Intent;
import android.net.Uri;
import android.os.Bundle;
import android.webkit.*;

public class MainActivity extends Activity {
    private WebView wv;
    private ValueCallback<Uri[]> fc;
    private static final int FR = 1;

    @Override
    protected void onCreate(Bundle s) {
        super.onCreate(s);
        wv = new WebView(this);
        setContentView(wv);
        WebSettings ws = wv.getSettings();
        ws.setJavaScriptEnabled(true);
        ws.setDomStorageEnabled(true);
        ws.setAllowFileAccess(true);
        ws.setAllowContentAccess(true);
        ws.setMixedContentMode(WebSettings.MIXED_CONTENT_ALWAYS_ALLOW);
        wv.setWebViewClient(new WebViewClient());
        wv.setWebChromeClient(new WebChromeClient() {
            @Override
            public boolean onShowFileChooser(WebView v, ValueCallback<Uri[]> cb, FileChooserParams p) {
                if (fc != null) fc.onReceiveValue(null);
                fc = cb;
                startActivityForResult(p.createIntent(), FR);
                return true;
            }
        });
        wv.loadUrl("file:///android_asset/index.html");
    }

    @Override
    protected void onActivityResult(int req, int res, Intent data) {
        if (req != FR || fc == null) return;
        Uri[] u = null;
        if (res == RESULT_OK && data != null) {
            String s = data.getDataString();
            if (s != null) u = new Uri[]{Uri.parse(s)};
        }
        fc.onReceiveValue(u);
        fc = null;
    }

    @Override
    public void onBackPressed() {
        if (wv.canGoBack()) wv.goBack();
        else super.onBackPressed();
    }
}
JAVA

cat > $APP/app/src/main/res/values/strings.xml << 'XML'
<?xml version="1.0" encoding="utf-8"?>
<resources><string name="app_name">BillVault</string></resources>
XML

cat > $APP/app/build.gradle << 'GRADLE'
plugins { id 'com.android.application' }
android {
    compileSdk 33
    defaultConfig {
        applicationId "com.billvault"
        minSdk 21
        targetSdk 33
        versionCode 20
        versionName "20.0"
    }
    buildTypes { release { minifyEnabled false } }
    compileOptions {
        sourceCompatibility JavaVersion.VERSION_1_8
        targetCompatibility JavaVersion.VERSION_1_8
    }
}
dependencies {}
GRADLE

cat > $APP/build.gradle << 'GRADLE'
buildscript {
    repositories { google(); mavenCentral() }
    dependencies { classpath 'com.android.tools.build:gradle:7.4.2' }
}
allprojects { repositories { google(); mavenCentral() } }
GRADLE

echo "include ':app'" > $APP/settings.gradle

cat > $APP/gradle/wrapper/gradle-wrapper.properties << 'PROPS'
distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\://services.gradle.org/distributions/gradle-7.6-bin.zip
PROPS

if [ ! -f "$APP/gradlew" ]; then
  wget -q https://raw.githubusercontent.com/gradle/gradle/v7.6.0/gradlew -O $APP/gradlew
  chmod +x $APP/gradlew
fi

echo "Setup complete!"
echo "index.html size: $(ls -lh $APP/app/src/main/assets/index.html | awk '{print $5}')"
echo "Ab Cell 4 run karo — APK build hoga (5-10 min)"


In [ ]:
%%bash
# Step 4: APK Build (5-10 minutes)
export ANDROID_HOME=/content/android-sdk
export ANDROID_SDK_ROOT=/content/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools:$ANDROID_HOME/build-tools/33.0.0

cd /content/BillVaultApp
chmod +x gradlew

echo "Building BillVault FINAL APK..."
./gradlew assembleDebug --no-daemon 2>&1 | tail -50

APK=app/build/outputs/apk/debug/app-debug.apk
if [ -f "$APK" ]; then
  SIZE=$(du -k "$APK" | cut -f1)
  echo ""
  echo "================================"
  echo "BUILD SUCCESS! Size: ${SIZE} KB"
  echo "================================"
  echo "Ab Cell 5 run karo"
else
  echo "BUILD FAILED — upar error dekho"
fi


In [ ]:
# Step 5: APK Download
from google.colab import files
import shutil, os

SRC = '/content/BillVaultApp/app/build/outputs/apk/debug/app-debug.apk'
DST = '/content/BillVault_FINAL.apk'

if os.path.exists(SRC):
    shutil.copy(SRC, DST)
    sz = os.path.getsize(DST)
    print(f"BillVault_FINAL.apk ready! ({sz/1e6:.1f} MB)")
    print()
    print("Features:")
    print("  AI: Gemini 2.5 (auto model discovery)")
    print("  Bills: Electricity, Property Tax, Water, LIC Premium")
    print("  LIC Members: configurable in the app")
    print("  Sheets: Google Drive upload + Photo Link in sheet")
    print("  Sheets URL: bound to your bills spreadsheet")
    print()
    print("Downloading...")
    files.download(DST)
else:
    print("APK nahi mili — Cell 4 run karo pehle")
    if os.path.exists('/content/BillVaultApp/app/build'):
        import subprocess
        r = subprocess.run(['find','/content/BillVaultApp/app/build','-name','*.apk'], capture_output=True, text=True)
        print("APK files found:", r.stdout)
